## 1. Extração (Camada Bronze)
Importação da base de litígios de consumo e identificação inicial dos problemas de tipagem e valores nulos.

In [ ]:
import pandas as pd
import numpy as np

# Carrega os dados brutos
df_bronze = pd.read_csv('../01_bronze/reclamacoes_raw.csv')

# Exibe as informações de tipagem e nulos
display(df_bronze.info())
display(df_bronze.head())

## 2. Tratamento de Strings e Entidades (Camada Silver)
Unificação dos diferentes nomes de registro das partes processuais (réus) e correção de espaços em branco.

In [ ]:
df_silver = df_bronze.copy()

# Remove espaços vazios e padroniza a UF para maiúsculo
df_silver['uf'] = df_silver['uf'].str.strip().str.upper()

# Padroniza os nomes das empresas para agrupar corretamente as métricas
df_silver['empresa'] = df_silver['empresa'].str.strip().str.upper()

# Dicionário de padronização de partes (réus comuns em varas cíveis/consumidor)
mapeamento_empresas = {
    'AMERICAN AIRLINES LTDA': 'AMERICAN AIRLINES',
    'MAIS CREDIT': 'MAISCREDIT',
    'MAISCREDIT CONSULTORIA': 'MAISCREDIT',
    'SÃO PAULO PREVIDÊNCIA': 'SPPREV',
    'UNIV. SÃO JUDAS': 'UNIVERSIDADE SÃO JUDAS'
}

df_silver['empresa'] = df_silver['empresa'].replace(mapeamento_empresas)

display(df_silver['empresa'].value_counts())

## 3. Conversão de Datas
A coluna `data_registro` tem múltiplos formatos estruturais (ex: brasileiro com barras, ISO com traços). A conversão para o tipo nativo `datetime` é fundamental para habilitar análises de inteligência de tempo (YTD, MoM) no Power BI.

In [ ]:
# O parâmetro dayfirst garante que 01/02/2026 seja lido como 1º de fevereiro, e format='mixed' lida com as variações
df_silver['data_registro'] = pd.to_datetime(df_silver['data_registro'], format='mixed', dayfirst=True)

display(df_silver[['id_chamado', 'data_registro']].head())

## 4. Tratamento numérico e de nulos
Resolução de anomalias na nota do consumidor (strings inseridas por erro humano e valores fora da escala de 1 a 5) e preenchimento de status vazios.

In [ ]:
# Preenche os status em branco com um valor rastreável
df_silver['status_resolucao'] = df_silver['status_resolucao'].fillna('Não Informado')

# Substitui o erro de digitação por extenso para número e converte a coluna para float
df_silver['nota_consumidor'] = df_silver['nota_consumidor'].replace('Dez', 10)
df_silver['nota_consumidor'] = pd.to_numeric(df_silver['nota_consumidor'])

# Trata outliers (notas fora do padrão de 1 a 5) substituindo-os por nulo para não distorcer a média
df_silver.loc[(df_silver['nota_consumidor'] < 1) | (df_silver['nota_consumidor'] > 5), 'nota_consumidor'] = np.nan

display(df_silver.describe())

## 5. Exportação (Camada Silver finalizada)
O dataset higienizado é exportado para alimentar o banco de dados relacional, onde será modelado em Fato e Dimensões.

In [ ]:
# Salva o arquivo limpo, garantindo que o index não seja exportado como uma coluna extra
caminho_saida = 'reclamacoes_silver.csv'
df_silver.to_csv(caminho_saida, index=False, encoding='utf-8')

print(f"Pipeline Silver concluído. Dados salvos em: {caminho_saida}")